# Create Sample DataSet for LLM

**Aim**
* from retrieved dataset, consolidate a final sample set with sufficient diversity per provider across 
    - base category key
    - BC (spend/non-spend)
    - category type
    - merchant_name
    - normalised transaction description
    - transaction description?


In [ ]:
from dotenv import load_dotenv
import os, gzip
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.csv as pacsv
import pandas as pd
from databricks import sql
import os
import plotly.express as px
import yaml
import re
import plotly.express as px
import numpy as np
import re
from collections import defaultdict

In [ ]:
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=pd.errors.SettingWithCopyWarning)

### Config and mappings

In [ ]:
COL = {
    'provider':           'provider_name',
    'spend':              'prop_ideas_has_merch',                        # 1=spend, 0=non-spend
    'category_type':      'prop_ideas_categorisation_name_nonspend',     # non-spend only
    'base_category_key':  'base_category_key',
    'merchant':           'prop_ideas_merchant_name',
    'description':        'original_description',
    'txn_type':           'transaction_type',
    'txn_id':             'transaction_id',
}

In [ ]:
SAMPLE_CONFIG = {
    'target_per_provider': 2000,

    # Spend vs non-spend split
    'spend_split': {
        '1': 0.50,    # spend
        '0': 0.50,    # non-spend
    },

    # Category type proportions — non-spend ONLY (has_merch=0)
    # Relative weights, normalised at runtime
    'category_type_non_spend': {
        'TRANSFERS':            0.33, #+0.03
        'ROUND_UP':             0.06, #-0.06
        'INTERESTS':            0.06, #-0.06
        'FEES':                 0.06, #-0.06
        'CREDIT_CARD_PAYMENT':  0.15, #+0.05
        'CASH':                 0.13, #+0.05
        'BENEFITS':             0.13, #+0.05
        'NOTES':                0.08,
    },

    # Cap on UNCATEGORISED within spend's base_category_key round-robin
    # Set to None to disable capping
    'spend_uncategorised_cap_pct': 0.15,

    # Ensure at least this many of each transaction_type per provider
    'min_per_transaction_type': 5,

    # Provider-specific overrides
    'provider_overrides': {
        # 'SomeSmallProvider': {'target_per_provider': 500},
    },

    # Description normalisation patterns
    'description_strip_patterns': [
        r'\d{3,}',
        r'\d{2}/\d{2}/\d{2,4}',
        r'\s+',
    ],
}

### Functions

In [ ]:
def normalise_description(desc, patterns=SAMPLE_CONFIG['description_strip_patterns']):
    if pd.isna(desc):
        return '_EMPTY_'
    s = str(desc).upper().strip()
    s = re.sub(patterns[0], '*', s)      # mask digit sequences
    s = re.sub(patterns[1], '*', s)      # mask dates
    s = re.sub(patterns[2], ' ', s)      # collapse whitespace
    return s.strip() or '_EMPTY_'


def normalise_weights(weight_dict):
    total = sum(weight_dict.values())
    return {k: v / total for k, v in weight_dict.items()}

def round_robin_sample(df_group, n, tier_columns):
    """Round-robin sample across tier_columns in priority order."""
    if n <= 0 or len(df_group) == 0:
        return df_group.iloc[:0]
    if len(df_group) <= n or len(tier_columns) == 0:
        return df_group.head(n)

    col = tier_columns[0]
    rng = np.random.RandomState(42)
    unique_vals = df_group[col].dropna().unique().tolist()
    rng.shuffle(unique_vals)

    sub_groups = {}
    for val in unique_vals:
        sub_groups[val] = df_group[df_group[col] == val].sample(frac=1, random_state=42)

    selected_indices = []
    pointers = {val: 0 for val in unique_vals}
    active_vals = list(unique_vals)

    while len(selected_indices) < n and active_vals:
        next_active = []
        for val in active_vals:
            if len(selected_indices) >= n:
                break
            sub = sub_groups[val]
            ptr = pointers[val]
            if ptr < len(sub):
                selected_indices.append(sub.index[ptr])
                pointers[val] = ptr + 1
                if ptr + 1 < len(sub):
                    next_active.append(val)
        active_vals = next_active

    return df_group.loc[selected_indices]


def sample_spend(df_spend, n, config):
    """Sample spend txns: optional UNCATEGORISED cap + round-robin diversity."""
    c = COL
    cap_pct = config.get('spend_uncategorised_cap_pct')

    if cap_pct is not None:
        cap_n = int(n * cap_pct)
        df_uncat = df_spend[df_spend[c['base_category_key']] == 'UNCATEGORISED']
        df_other = df_spend[df_spend[c['base_category_key']] != 'UNCATEGORISED']

        s_uncat = round_robin_sample(df_uncat, cap_n, [c['merchant'], 'norm_desc'])
        s_other = round_robin_sample(df_other, n - len(s_uncat), [c['base_category_key'], c['merchant'], 'norm_desc'])
        return pd.concat([s_uncat, s_other], ignore_index=True)
    else:
        return round_robin_sample(df_spend, n, [c['base_category_key'], c['merchant'], 'norm_desc'])


def sample_non_spend(df_non_spend, n, config):
    """Sample non-spend: category_type quotas + round-robin diversity."""
    c = COL
    ct_weights = normalise_weights(config.get('category_type_non_spend', {'_OTHER': 1.0}))
    actual_types = df_non_spend[c['category_type']].unique().tolist()

    assigned = defaultdict(list)
    for ct in actual_types:
        assigned[ct if ct in ct_weights else '_OTHER'].append(ct)

    raw_quotas = {}
    for ck, weight in ct_weights.items():
        mapped = assigned.get(ck, [])
        if mapped:
            quota = max(1, int(n * weight))
            per_type = max(1, quota // len(mapped))
            for ct in mapped:
                raw_quotas[ct] = per_type

    available = {ct: len(df_non_spend[df_non_spend[c['category_type']] == ct]) for ct in raw_quotas}

    quotas, surplus, uncapped = {}, 0, []
    for ct, quota in raw_quotas.items():
        avail = available.get(ct, 0)
        if avail < quota:
            quotas[ct] = avail
            surplus += quota - avail
        else:
            quotas[ct] = quota
            uncapped.append(ct)

    if surplus > 0 and uncapped:
        extra_each = surplus // len(uncapped)
        remainder = surplus % len(uncapped)
        for i, ct in enumerate(uncapped):
            quotas[ct] = min(quotas[ct] + extra_each + (1 if i < remainder else 0), available.get(ct, 0))

    parts = []
    for ct, quota in quotas.items():
        df_s = df_non_spend[df_non_spend[c['category_type']] == ct]
        if len(df_s) > 0:
            parts.append(round_robin_sample(df_s, quota, [c['base_category_key'], c['merchant'], 'norm_desc']))

    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()


def sample_provider(df_provider, config):
    """Full sampling pipeline for one provider."""
    c = COL
    provider = df_provider[c['provider']].iloc[0]
    overrides = config.get('provider_overrides', {}).get(provider, {})
    cfg = {**config, **overrides}
    target_n = cfg.get('target_per_provider', 2000)

    df_provider = df_provider.copy()
    df_provider['norm_desc'] = df_provider[c['description']].apply(normalise_description)

    spend_n = int(target_n * cfg['spend_split'].get('1', 0.5))
    non_spend_n = target_n - spend_n

    s_spend = sample_spend(df_provider[df_provider[c['spend']] == '1'], spend_n, cfg)
    s_non_spend = sample_non_spend(df_provider[df_provider[c['spend']] == '0'], non_spend_n, cfg)
    df_sampled = pd.concat([s_spend, s_non_spend], ignore_index=True)

    # Backfill: if non-spend couldn't fill its quota, give leftover slots to spend
    shortfall = target_n - len(df_sampled)
    if shortfall > 0:
        remaining = df_provider[
            (df_provider[c['spend']] == '1') &
            (~df_provider.index.isin(df_sampled.index))
        ]
        backfill = round_robin_sample(remaining, shortfall, [c['base_category_key'], c['merchant'], 'norm_desc'])
        df_sampled = pd.concat([df_sampled, backfill], ignore_index=True)

    # Transaction type top-up
    min_tt = cfg.get('min_per_transaction_type', 5)
    if len(df_sampled) < target_n:
        tt_counts = df_sampled[c['txn_type']].value_counts()
        top_ups = []
        for tt in df_provider[c['txn_type']].unique():
            if tt_counts.get(tt, 0) < min_tt:
                cands = df_provider[(df_provider[c['txn_type']] == tt) & (~df_provider.index.isin(df_sampled.index))]
                top_ups.append(cands.head(min_tt - tt_counts.get(tt, 0)))
        if top_ups:
            df_sampled = pd.concat([df_sampled] + top_ups, ignore_index=True).head(target_n)

    return df_sampled


def validation_report(df_v, config, provider=None):
    """Print distribution comparison for sampled data."""
    c = COL
    if provider:
        df_v = df_v[df_v[c['provider']] == provider]
    label = provider or 'ALL PROVIDERS'
    n = len(df_v)
    if n == 0:
        print(f"\n[{label}] No data"); return

    print(f"\n{'═'*60}")
    print(f"  {label}  (n={n:,})")
    print(f"{'═'*60}")

    # Spend split
    print("\n  Spend / Non-Spend:")
    spend_actual = df_v[c['spend']].value_counts(normalize=True) * 100
    for k, target in config['spend_split'].items():
        lbl = 'spend' if k == '1' else 'non_spend'
        actual = spend_actual.get(k, 0)
        delta = actual - target * 100
        flag = '⚠️' if abs(delta) > 10 else '✅'
        print(f"    {lbl:12s}  target={target*100:5.1f}%  actual={actual:5.1f}%  Δ={delta:+.1f}%  {flag}")

    # Category type (non-spend only)
    ct_weights = normalise_weights(config.get('category_type_non_spend', {}))
    df_ns = df_v[df_v[c['spend']] == '0']
    if len(df_ns) > 0:
        print(f"\n  Category Types (non-spend):")
        for ct, pct in (df_ns[c['category_type']].value_counts(normalize=True) * 100).items():
            target = ct_weights.get(ct, ct_weights.get('_OTHER', 0)) * 100
            delta = pct - target
            flag = '⚠️' if abs(delta) > 10 else '✅'
            print(f"    {str(ct):25s}  actual={pct:5.1f}%  (target~{target:5.1f}%)  Δ={delta:+.1f}%  {flag}")

    # Top base_category_key for spend
    df_sp = df_v[df_v[c['spend']] == '1']
    if len(df_sp) > 0:
        print(f"\n  Top 10 base_category_key (spend):")
        for cat, pct in (df_sp[c['base_category_key']].value_counts(normalize=True).head(10) * 100).items():
            print(f"    {str(cat):30s}  {pct:5.1f}%{'  ← capped' if cat == 'UNCATEGORISED' else ''}")

    # Diversity
    print(f"\n  Diversity:")
    print(f"    base_category_key:  {df_v[c['base_category_key']].nunique()} unique")
    print(f"    merchant_name:      {df_v[c['merchant']].nunique()} unique")
    print(f"    transaction_type:   {df_v[c['txn_type']].nunique()} unique → {df_v[c['txn_type']].value_counts().to_dict()}")


def heatmap_crosstab(df, index_col, value_col, show_pct=True, height=600, width=1000, label_map=None):
    """Plotly heatmap from a crosstab."""
    ct = pd.crosstab(df[index_col], df[value_col])
    if label_map:
        ct = ct.rename(columns={k: f"{k} ({v})" for k, v in label_map.items()})
    data = ct.div(ct.sum(axis=1), axis=0) * 100 if show_pct else ct

    fig = px.imshow(
        data,
        text_auto='.1f' if show_pct else 'd',
        color_continuous_scale='YlOrRd',
        labels={'color': '%' if show_pct else 'count'},
        title=f'{value_col} by {index_col} ({"%" if show_pct else "counts"})',
        height=height, width=width, aspect='auto',
    )
    fig.update_xaxes(side='bottom', dtick=1, tickangle=0)
    fig.update_coloraxes(showscale=False)
    fig.show()

### a. Test on 1 provider

In [ ]:
df = pd.read_parquet("data/raw_data2_large.parquet")

In [ ]:
TEST_PROVIDER = 'ANZ'  # ← change this to test different providers

df_prov = df[df[COL['provider']] == TEST_PROVIDER]
df_s = sample_provider(df_prov, SAMPLE_CONFIG)

print(f"Available: {len(df_prov):,}  →  Sampled: {len(df_s):,}")
validation_report(df_s, SAMPLE_CONFIG, TEST_PROVIDER)


In [ ]:
#Eyeball the test sample — tweak as needed
df_s[COL['base_category_key']].value_counts().head(20)

In [ ]:
#Eyeball merchants
df_s[COL['merchant']].value_counts().head(20)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

df_prov[COL['base_category_key']].value_counts(normalize=True).head(30).plot.barh(ax=axes[0], title='BEFORE (raw)')
df_s[COL['base_category_key']].value_counts(normalize=True).head(30).plot.barh(ax=axes[1], title='AFTER (sampled)', color='orange')

axes[0].set_xlabel('%')
axes[1].set_xlabel('%')
plt.tight_layout()
plt.show()

#### create final sample set

In [ ]:
df_sampled = pd.concat(
    [sample_provider(df[df[COL['provider']] == p], SAMPLE_CONFIG) for p in df[COL['provider']].unique()],
    ignore_index=True
)

print(f"Total sampled: {len(df_sampled):,}")

In [ ]:
df_report = (
    df_sampled.groupby(COL['provider'])
    .agg(
        sampled=(COL['txn_id'], 'count'),
        spend_pct=(COL['spend'], lambda x: (x == '1').mean() * 100),
        unique_base_cat=(COL['base_category_key'], 'nunique'),
        unique_merchants=(COL['merchant'], 'nunique'),
        unique_txn_types=(COL['txn_type'], 'nunique'),
    )
    .reset_index()
    .sort_values('sampled', ascending=False)
)
print(df_report.to_string(index=False))

In [ ]:
## 11. per provider validation
for p in sorted(df_sampled[COL['provider']].unique()):
    validation_report(df_sampled, SAMPLE_CONFIG, p)


In [ ]:
# 12 Before / after heatmap - all provider
print("BEFORE:")
heatmap_crosstab(df[df[COL['spend']] == '0'], COL['provider'], COL['category_type'], height=700, width=1000)

print("AFTER:")
heatmap_crosstab(df_sampled[df_sampled[COL['spend']] == '0'], COL['provider'], COL['category_type'], height=700, width=1000)

In [ ]:
#13 - Transaction type comparison

TXN_TYPE_MAP = {0:'fee', 1:'interest_charged', 2:'interest_paid', 3:'transfer_outgoing', 4:'transfer_incoming', 5:'payment', 6:'direct_debit', 7:'other'}

print("BEFORE:")
heatmap_crosstab(df, COL['provider'], COL['txn_type'], label_map=TXN_TYPE_MAP, width=1200)

print("AFTER:")
heatmap_crosstab(df_sampled, COL['provider'], COL['txn_type'], label_map=TXN_TYPE_MAP, width=1200)

In [ ]:
df_sampled.shape

In [ ]:
df_sampled.to_parquet(
    "data/sample_data3.parquet",
    engine="pyarrow",       # or "fastparquet"
    compression="snappy",   # common: "snappy", "gzip", "zstd", or None
    index=False             # often desirable to avoid saving the index
)


In [ ]:
df_sampled.head().T